# rl-v1 -- cluster-rl-training, parallel tabular Q-learning on all training seeds

Runs `train_cluster_rl.py` (repo root), which extracts `cluster-rl-training.zip` to
`~/cluster-rl-training`, installs its requirements, runs `cluster.py setup` + the package unit
tests, initializes `runs/all-seeds` from the six-game warm start, trains `ROUNDS` shared-learner
rounds over **every training seed** (`TRAIN_SEEDS`, one full normal game per seed per round,
`WORKERS` games at a time), and after each round evaluates that round's checkpoint against the baseline on the
validation seeds (frozen, no exploration) so a checkpoint can be selected on validation.

CPU only. Rerunning the training cell resumes: finished games/merges/evaluations are reused.
Test seeds (3000:3032) are reserved and never touched here.

**Cluster setup:** same as `test-eat-rest-v1.ipynb` -- a git-ignored `.env` with
`GITHUB_TOKEN=<token>` in the kernel's starting directory.

In [4]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [5]:
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 12 (delta 7), reused 12 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.55 KiB | 1.27 MiB/s, done.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
   fb083b3..23610c7  challenge-1V2 -> origin/challenge-1V2
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is behind 'origin/challenge-1V2' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Updating fb083b3..23610c7
Fast-forward
 .../survival-simulator/.gitattributes              |  1 +
 .../survival-simulator/deploy/push.sh              | 15 +++++
 .../survival-simulator/submission_server.py        | 66 +++++++++++++---------
 3 files chan

In [ ]:
import time

TRAIN_SEEDS = "1000:1256"        # start-inclusive, stop-exclusive: all 256 package-default training seeds
VALIDATION_SEEDS = "2000:2016"
TEST_SEEDS = "3000:3032"         # reserved, not run by this notebook
ROUNDS = 10                      # total rounds incl. completed ones (256 games each); raise and rerun to continue
WORKERS = 0                      # 0 = auto: min(container CPU limit, (free RAM - 2 GiB) / 2 GiB per game); CPU-bound, the GPU is not used
RUN = "runs/all-seeds"           # fixed once initialized; use a new name for different seeds

cmd = [sys.executable, "-u", "train_cluster_rl.py", "--run", RUN, "--train-seeds", TRAIN_SEEDS,
       "--validation-seeds", VALIDATION_SEEDS, "--test-seeds", TEST_SEEDS,
       "--rounds", str(ROUNDS), "--workers", str(WORKERS)]
print("running:", " ".join(cmd))

t0 = time.perf_counter()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed = time.perf_counter() - t0

print(f"finished in {elapsed / 60:.1f} min, exit code {proc.returncode}")
if proc.returncode != 0:
    raise RuntimeError(f"train_cluster_rl.py failed with exit code {proc.returncode}")

running: /opt/conda/bin/python -u train_cluster_rl.py --run runs/all-seeds --train-seeds 1000:1256 --validation-seeds 2000:2016 --test-seeds 3000:3032 --rounds 10 --workers 0
resources: 40 usable CPUs, 268.2 GiB free RAM -> 40 concurrent games (cpu limit 40, ram limit 133 at 2 GiB/game)
running: /opt/conda/bin/python -m pip install -q -r requirements.txt
running: /opt/conda/bin/python /home/jovyan/cluster-rl-training/cluster.py setup
{"simulator_fingerprint": "5b85c917c5ae45a30789a01b70411d05ccbe0f19d43ab4a381e9afbc9b9a3ffd"}
running: /opt/conda/bin/python -m unittest discover -s tests
.......{"event": "merged", "checkpoint": "/tmp/tmpfob5zbe3/checkpoints/round_0000.json", "new_transitions": 1, "states": 1}
{"event": "checkpoint_reused", "checkpoint": "/tmp/tmpfob5zbe3/checkpoints/round_0000.json"}
....
----------------------------------------------------------------------
Ran 11 tests in 0.355s

OK
/home/jovyan/cluster-rl-training/runs/all-seeds already initialized - resuming with its

In [ ]:
import json
import pandas as pd

RUN_DIR = os.path.join(os.path.expanduser("~"), "cluster-rl-training", RUN)
summaries = json.load(open(os.path.join(RUN_DIR, "validation_summary.json")))

# One row per checkpoint: frozen validation score vs the baseline on the same seeds
per_round = pd.DataFrame([dict(checkpoint=name, mean=s["mean"], median=s["median"], minimum=s["minimum"],
                               full_horizon=s["full_horizon"], baseline_mean=s["paired"]["baseline_mean"],
                               mean_delta=s["paired"]["mean_delta"], seeds_better=s["paired"]["seeds_better"])
                          for name, s in sorted(summaries.items())])
per_round

In [ ]:
# Per-seed paired table for the checkpoint with the best validation mean
best = per_round.sort_values("mean", ascending=False).iloc[0]["checkpoint"]
s = summaries[best]
df = pd.DataFrame({"hybrid": s["seeds"], "delta_vs_baseline": s["paired"]["differences"]})
df.index.name = "seed"
df["baseline"] = df["hybrid"] - df["delta_vs_baseline"]
print("best on validation:", best)
df[["baseline", "hybrid", "delta_vs_baseline"]]

In [ ]:
# Per-round training survival (training games explore, so this is NOT evidence of improvement)
import glob
rows = []
for path in sorted(glob.glob(os.path.join(RUN_DIR, "checkpoints", "round_*.json"))):
    ck = json.load(open(path))
    s = pd.Series([e["survival"] for e in ck["episodes"]])
    rows.append(dict(round=ck["round"], games=len(s), mean=s.mean(), median=s.median(), min=s.min(),
                     new_transitions=ck["added_transitions"], states=len(ck["model"]["q"])))
pd.DataFrame(rows)